In [1]:
# imports   
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt
import ta
import datetime

In [ ]:
# load data from csv 
original_data = pd.read_csv('../data/processed/BTCUSDT_1m_2024-12-01_to_2025-01-01_cleaned_robust.csv')
volume_data = pd.read_csv('../data/processed/BTCUSDT_1m_2024-12-01_to_2025-01-volume.csv')
dolar_data = pd.read_csv('../data/processed/BTCUSDT_1m_2024-12-01_to_2025-01-01_dollar_bars_dyn.csv')

In [3]:
original_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43138 entries, 0 to 43137
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   open_time        43138 non-null  object 
 1   open             43138 non-null  float64
 2   high             43138 non-null  float64
 3   low              43138 non-null  float64
 4   close            43138 non-null  float64
 5   volume           43138 non-null  float64
 6   close_time       43138 non-null  object 
 7   quote_volume     43138 non-null  float64
 8   trades           43138 non-null  float64
 9   taker_buy_base   43138 non-null  float64
 10  taker_buy_quote  43138 non-null  float64
 11  ignore           43138 non-null  float64
 12  mid_price        43138 non-null  float64
 13  return           43138 non-null  float64
dtypes: float64(12), object(2)
memory usage: 4.6+ MB


In [4]:
volume_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5286 entries, 0 to 5285
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   open_time   5286 non-null   object 
 1   open        5286 non-null   float64
 2   high        5286 non-null   float64
 3   low         5286 non-null   float64
 4   close       5286 non-null   float64
 5   volume      5286 non-null   float64
 6   close_time  5286 non-null   object 
 7   return      5285 non-null   float64
dtypes: float64(6), object(2)
memory usage: 330.5+ KB


In [5]:
dolar_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29397 entries, 0 to 29396
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   open_time     29397 non-null  object 
 1   close_time    29397 non-null  object 
 2   open          29397 non-null  float64
 3   high          29397 non-null  float64
 4   low           29397 non-null  float64
 5   close         29397 non-null  float64
 6   volume        29397 non-null  float64
 7   quote_volume  29397 non-null  float64
 8   dollar_value  29397 non-null  float64
 9   mid_price     29397 non-null  float64
 10  num_ticks     29397 non-null  int64  
 11  return        29397 non-null  float64
dtypes: float64(9), int64(1), object(2)
memory usage: 2.7+ MB


Para el modelo predictivo hay que tener las mismas features, así que hay que revisar los notebooks, donde se trataron los datos, también hay que tener en cuenta las features que usamos en un anterior proyecto apra darle más info al modelo, están en obsidian explicadas

Hay que quitar lo de close_time, open_time, tienen directamente usarlas como features, al calcular los retornos hay que usar shift(1) correctaemtne para solo usar los datos anteriores al target.

Los features bases para cada dataset son los siguientes, hay que quitarlos dentro del tratamiento de los bars correspondientes dentro de los notebooks donde se limpiaron los datos. Las features son las siguientes: 

['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'return']


In [ ]:
# date object to datetime

def init_date_conversion(data):
    print("dtype original:", data['open_time'].dtype)
    print("primeros 10 valores (raw):")
    display(data['open_time'].head(10))

    if data['open_time'].dtype == object:
        # eliminar espacios, comillas, saltos de línea
        data['open_time'] = data['open_time'].astype(str).str.strip().str.replace('"', '', regex=False).str.replace("'", "", regex=False)
        data['close_time'] = data['close_time'].astype(str).str.strip().str.replace('"', '', regex=False).str.replace("'", "", regex=False)

    sample_vals = data['open_time'].dropna().astype(str).head(50)
    n_digit_samples = sample_vals.apply(lambda x: x.isdigit()).sum()
    print(f"de los primeros 50 valores, {n_digit_samples} parecen solo dígitos (epoch)")

def robust_to_datetime(series):
    # si ya es datetime, devolver
    if np.issubdtype(series.dtype, np.datetime64):
        return series
    
    s = series.copy()
    as_str = s.dropna().astype(str)
    is_all_digits = as_str.str.match(r'^\d+$').mean()  # proporción de strings sólo dígitos
    
    if is_all_digits > 0.5:
        # la mayoría son timestamps numéricos; inferir ms vs s por magnitud
        # convertir a float para inspección del tamaño (usa sample para evitar overflow)
        sample_num = as_str.sample(min(100, len(as_str))).astype(float)
        median_sample = sample_num.median()
        print("mediana de muestra numérica:", median_sample)
        if median_sample > 1e12:  # típico de ms epoch ( > ~10^12)
            print("-> inferido como epoch en MILLISEGUNDOS")
            out = pd.to_datetime(s.astype(float), unit='ms', errors='coerce')
        elif median_sample > 1e9:  # típico de s epoch ( > ~10^9)
            print("-> inferido como epoch en SEGUNDOS")
            out = pd.to_datetime(s.astype(float), unit='s', errors='coerce')
        else:
            # números pequeños: tratar como strings normalmente
            out = pd.to_datetime(s, errors='coerce')
        return out
    else:
        out = pd.to_datetime(s, errors='coerce', utc=False)
        # si demasiados NaT, intentar forzar formatos comunes
        nat_frac = out.isna().mean()
        print(f"frac NaT tras parse directo: {nat_frac:.3f}")
        if nat_frac > 0.2:
            # intentar parse con varias plantillas comunes
            fmts = [
                "%Y-%m-%d %H:%M:%S.%f",
                "%Y-%m-%d %H:%M:%S",
                "%Y-%m-%dT%H:%M:%S.%fZ",
                "%Y-%m-%dT%H:%M:%SZ",
            ]
            for fmt in fmts:
                try:
                    test = pd.to_datetime(s, format=fmt, errors='coerce')
                    nat_frac2 = test.isna().mean()
                    print(f"  intento con format {fmt} => NaT frac: {nat_frac2:.3f}")
                    if nat_frac2 < nat_frac:
                        out = test
                        nat_frac = nat_frac2
                except Exception:
                    pass
        return out

# Aplicar la función a open_time y close_time
def convert_datetime_columns(data):
    data['open_time_parsed'] = robust_to_datetime(data['open_time'])
    data['close_time_parsed'] = robust_to_datetime(data['close_time'])

    n_total = len(data)
    n_nat_open = data['open_time_parsed'].isna().sum()
    n_nat_close = data['close_time_parsed'].isna().sum()
    print(f"open_time -> NaT: {n_nat_open}/{n_total} ({n_nat_open/n_total:.3%})")
    print(f"close_time -> NaT: {n_nat_close}/{n_total} ({n_nat_close/n_total:.3%})")

    if n_nat_open > 0:
        print("Ejemplos de open_time problemáticos:")
        display(data.loc[data['open_time_parsed'].isna(), 'open_time'].head(10))


    if (n_nat_open / n_total) < 0.05:
        data['open_time'] = data['open_time_parsed']
    else:
        print("Advertencia: demasiados NaT en open_time — revisa los ejemplos anteriores.")
        # no sobrescribimos para evitar pérdida de info

    if (n_nat_close / n_total) < 0.05:
        data['close_time'] = data['close_time_parsed']

    # borrar columnas auxiliares
    data = data.drop(columns=[c for c in ['open_time_parsed', 'close_time_parsed'] if c in data.columns])


    data = data.dropna(subset=['open_time'])

    print("Conversión final: dtype open_time =>", data['open_time'].dtype)
    display(data.head())

    return pd.DataFrame(data)


In [ ]:
# delete columns (close_time, open_time) 3 datasets

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # convert existent times 
    if 'open_time' in df.columns and 'close_time' in df.columns:
        df = convert_datetime_columns(df)
        df['bar_duration'] = (df['close_time'] - df['open_tiem']).dt.total_seconds()
    else: 
        df['bar_duration'] = np.nan